# G1_03 — Database e injection

> **Corso ITS D.E.Mo.S. — Laboratorio API + Frontend — Giornata 1**
> Esegui le celle dall'alto verso il basso con **Shift+Invio**. Prima di ogni cella c'è scritto **cosa aspettarti**.
> Se qualcosa non torna, alza la mano: l'errore si legge insieme.

## Cosa faremo (60 minuti)

L'API di stamattina **ricorda** i ticket anche se il server si riavvia. Come? Li salva in un database.
Useremo **SQLite**: un database intero dentro un solo file, già incluso in Python. Oggi pomeriggio lo userai nell'API.

Poi faremo una cosa che di solito non si fa a lezione: **bucheremo la nostra stessa query** con una SQL injection.
Vedere l'attacco funzionare è il modo più sicuro per non scrivere mai più codice vulnerabile.

### Cella 1 — Aprire un database

`sqlite3.connect("file.db")` apre (o crea) il file. Il **cursore** è l'oggetto con cui si mandano i comandi SQL.

**Cosa aspettarti:** "Database aperto".

In [ ]:
import sqlite3

connection = sqlite3.connect("tickets.db")   # crea il file se non esiste
connection.row_factory = sqlite3.Row         # per leggere le colonne per nome
cursor = connection.cursor()

print("Database aperto:", "tickets.db")

### Cella 2 — Creare la tabella

Una tabella è come un foglio Excel: colonne con un nome e un tipo, righe con i dati.
`IF NOT EXISTS` evita l'errore se la cella viene eseguita due volte.

**Cosa aspettarti:** "Tabella pronta".

In [ ]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS tickets (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        title       TEXT NOT NULL,
        description TEXT NOT NULL DEFAULT '',
        status      TEXT NOT NULL DEFAULT 'aperto'
    )
""")
connection.commit()      # rende definitive le modifiche

print("Tabella pronta")

### Cella 3 — Inserire righe

`INSERT` aggiunge una riga. I valori li passiamo **a parte**, come tupla, e nella query mettiamo dei `?`.
Tienilo a mente: tra poco capirai perché è importante.

**Cosa aspettarti:** "Inseriti 3 ticket".

In [ ]:
sample_tickets = [
    ("Stampante del secondo piano non stampa", "Coda ferma", "aperto"),
    ("Password scaduta", "Non accedo al gestionale", "in_lavorazione"),
    ("Monitor che sfarfalla", "Postazione 14", "chiuso"),
]

for title, description, status in sample_tickets:
    cursor.execute(
        "INSERT INTO tickets (title, description, status) VALUES (?, ?, ?)",
        (title, description, status),
    )
connection.commit()

print("Inseriti", len(sample_tickets), "ticket")

### Cella 4 — Leggere le righe

`SELECT` legge. `fetchall()` restituisce tutte le righe trovate.

**Cosa aspettarti:** 3 righe, una per ticket, con id crescente.

In [ ]:
rows = cursor.execute("SELECT * FROM tickets ORDER BY id").fetchall()

for row in rows:
    print(f"#{row['id']}  [{row['status']:<15}] {row['title']}")

### Cella 5 — Filtrare: la funzione che useremo nell'API

Nell'API il filtro `?status=aperto` diventa una `SELECT ... WHERE status = ...`.
Scriviamola come funzione, nel modo **sbagliato ma comune**: costruendo la query con una f-string.

**Cosa aspettarti:** funziona. Torna solo il ticket aperto. Sembra tutto a posto.

In [ ]:
def find_by_status_UNSAFE(status: str):
    query = f"SELECT * FROM tickets WHERE status = '{status}'"   # <- il valore finisce DENTRO la query
    print("Query eseguita:", query)
    return cursor.execute(query).fetchall()


for row in find_by_status_UNSAFE("aperto"):
    print(f"#{row['id']}  {row['title']}")

### Cella 6 — L'attacco: SQL injection

Il valore di `status` arriva **dall'utente**. E se l'utente non scrive `aperto` ma scrive questo?

```
' OR '1'='1
```

Guarda la query stampata: l'apice chiude la stringa, e `OR '1'='1'` è sempre vero. **Escono tutti i ticket**, anche quelli che non dovevi vedere.

**Cosa aspettarti:** tutti e 3 i ticket invece di uno solo.

In [ ]:
malicious_input = "' OR '1'='1"

rows = find_by_status_UNSAFE(malicious_input)
print()
print("Righe ottenute:", len(rows), "(dovevano essere 0: nessun ticket ha quello stato!)")
for row in rows:
    print(f"#{row['id']}  {row['title']}")

### Cella 7 — Peggio: cancellare la tabella

Con `;` si chiude un comando e se ne inizia un altro. `sqlite3` con `execute` esegue un solo comando alla volta,
ma altri database (e `executescript`) no. Vediamo cosa **succederebbe**.

**Cosa aspettarti:** la tabella `tickets` **non esiste più**. La ricreiamo subito dopo, ma pensa se fosse il database vero dell'azienda.

In [ ]:
malicious_input = "x'; DROP TABLE tickets; --"
query = f"SELECT * FROM tickets WHERE status = '{malicious_input}'"
print("Query costruita:", query)

cursor.executescript(query)     # esegue TUTTI i comandi separati da ';'

try:
    cursor.execute("SELECT COUNT(*) FROM tickets")
    print("La tabella c'e' ancora")
except sqlite3.OperationalError as error:
    print("ERRORE:", error)
    print("La tabella e' stata cancellata da una stringa mandata dall'utente.")

### Cella 8 — Ricostruiamo e ripariamo: i parametri `?`

Ricreiamo la tabella (riusando le celle 2 e 3, in forma compatta) e riscriviamo la funzione **nel modo giusto**:
la query ha un `?` e il valore viaggia **separato**, in una tupla. Il database sa che quello è un *valore*, non un pezzo di comando.
Qualsiasi apice o `;` dentro al valore viene trattato come semplice testo.

**Cosa aspettarti:** con `"aperto"` torna 1 riga; con la stringa malevola tornano **0 righe**. La tabella è intatta.

In [ ]:
# Ricostruzione
cursor.execute("""
    CREATE TABLE IF NOT EXISTS tickets (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        title TEXT NOT NULL,
        description TEXT NOT NULL DEFAULT '',
        status TEXT NOT NULL DEFAULT 'aperto'
    )
""")
cursor.executemany(
    "INSERT INTO tickets (title, description, status) VALUES (?, ?, ?)", sample_tickets
)
connection.commit()


# La versione SICURA
def find_by_status(status: str):
    query = "SELECT * FROM tickets WHERE status = ?"      # il '?' e' un segnaposto
    return cursor.execute(query, (status,)).fetchall()    # il valore viaggia a parte


print("Con 'aperto':          ", len(find_by_status("aperto")), "riga/e")
print("Con l'input malevolo:  ", len(find_by_status("' OR '1'='1")), "riga/e")
print("Con il DROP TABLE:     ", len(find_by_status("x'; DROP TABLE tickets; --")), "riga/e")
print()
print("Ticket ancora in tabella:", cursor.execute("SELECT COUNT(*) FROM tickets").fetchone()[0])

### Cella 9 — Modificare e cancellare (ti servono nel pomeriggio)

`UPDATE` cambia una riga, `DELETE` la toglie. Sempre con i `?`. `rowcount` dice quante righe sono state toccate:
se è 0, l'`id` non esisteva → nell'API risponderemo `404`.

**Cosa aspettarti:** il ticket 1 passa a `chiuso`; il ticket 999 non esiste (rowcount 0); il ticket 3 sparisce.

In [ ]:
# UPDATE
cursor.execute("UPDATE tickets SET status = ? WHERE id = ?", ("chiuso", 1))
print("Righe modificate:", cursor.rowcount)

cursor.execute("UPDATE tickets SET status = ? WHERE id = ?", ("chiuso", 999))
print("Righe modificate con id inesistente:", cursor.rowcount, "-> nell'API sara' un 404")

# DELETE
cursor.execute("DELETE FROM tickets WHERE id = ?", (3,))
print("Righe cancellate:", cursor.rowcount)
connection.commit()

print()
for row in cursor.execute("SELECT * FROM tickets ORDER BY id"):
    print(f"#{row['id']}  [{row['status']:<15}] {row['title']}")

### Cella 10 — Tocca a te

Scrivi la funzione `insert_ticket(title, description)` che inserisce un ticket con stato `aperto` **usando i parametri**
e restituisce l'`id` della riga creata (`cursor.lastrowid`).
Poi provala con un titolo "cattivo" come `"Rotto'; DROP TABLE tickets; --"` e verifica che venga salvato come **testo normale**.

In [ ]:
def insert_ticket(title: str, description: str) -> int:
    # completa qui: INSERT con ? e (title, description) come tupla
    ...
    connection.commit()
    return cursor.lastrowid


new_id = insert_ticket("Rotto'; DROP TABLE tickets; --", "titolo cattivo")
row = cursor.execute("SELECT * FROM tickets WHERE id = ?", (new_id,)).fetchone()
if row is None:
    print("La funzione non e' ancora completa: sostituisci i ... con l'INSERT")
else:
    print("Salvato come testo:", row["title"])
print("Tabella intatta, righe:", cursor.execute("SELECT COUNT(*) FROM tickets").fetchone()[0])

## Riepilogo

- SQLite: un database in un file, dentro Python. `connect`, `execute`, `commit`, `fetchall`.
- `CREATE TABLE`, `INSERT`, `SELECT`, `UPDATE`, `DELETE`: le cinque frasi che ti servono.
- **SQL injection**: se il valore dell'utente finisce *dentro* la stringa della query, l'utente può cambiare la query.
- **Difesa**: sempre `?` e tupla di parametri. **Mai** f-string, `+` o `.format()` per costruire SQL.
- Validazione (notebook 2) + parametri (notebook 3) = le due difese di base di ogni API.

Nel pomeriggio queste due difese le metti nella **tua** API, partendo dal template del docente.